# SQL na Prática: Um Caso de Crédito — Arquivo para consultas locais

Esse arquivo carrega os 3 CSVs (`data/customers.csv`, `data/credit_applications.csv`, `data/loans.csv`) em um banco DuckDB em memória e roda as mesmas queries do case, adaptadas para a sintaxe do DuckDB (arquivos em `queries_db/`).

As queries originais para BigQuery ficam em `queries_bq/`.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}" if abs(v) < 1 else f"{v:,.2f}")

con = duckdb.connect()

## Setup: criar as tabelas e carregar os CSVs

Os tipos abaixo espelham o DDL usado no BigQuery (`data/schema.sql`), só traduzidos para os tipos equivalentes do DuckDB (`INT64` → `BIGINT`, `FLOAT64` → `DOUBLE`, `STRING` → `VARCHAR`).

In [2]:
con.execute("""
CREATE OR REPLACE TABLE customers (
  customer_id BIGINT,
  created_at DATE,
  state VARCHAR,
  income DOUBLE,
  age BIGINT
);
""")

con.execute("""
CREATE OR REPLACE TABLE credit_applications (
  application_id BIGINT,
  customer_id BIGINT,
  application_date DATE,
  requested_amount DOUBLE,
  approved_amount DOUBLE,
  status VARCHAR,
  risk_score BIGINT
);
""")

con.execute("""
CREATE OR REPLACE TABLE loans (
  loan_id BIGINT,
  customer_id BIGINT,
  contract_date DATE,
  loan_amount DOUBLE,
  due_date DATE,
  payment_date DATE,
  amount_paid DOUBLE
);
""")

con.execute("COPY customers FROM 'data/customers.csv' (HEADER, DELIMITER ',')")
con.execute("COPY credit_applications FROM 'data/credit_applications.csv' (HEADER, DELIMITER ',')")
con.execute("COPY loans FROM 'data/loans.csv' (HEADER, DELIMITER ',')")

for tabela in ["customers", "credit_applications", "loans"]:
    n = con.sql(f"SELECT count(*) AS n FROM {tabela}").df().iloc[0, 0]
    print(f"{tabela}: {n} linhas")

customers: 10000 linhas
credit_applications: 16513 linhas
loans: 8218 linhas


## Helper para rodar as queries

Lê o `.sql` de `queries_db/` e mostra o resultado como tabela (DataFrame).

In [3]:
def run_query(nome_arquivo: str) -> pd.DataFrame:
    with open(f"queries_db/{nome_arquivo}") as f:
        sql = f.read()
    print(sql)
    return con.sql(sql).df()

## Parte A — Exploração e métricas básicas

### A.1 — Quantas solicitações de crédito foram realizadas por mês?

In [4]:
run_query("a-1-solicitacoes-mes.sql")

-- Parte A, questão 1: Quantas solicitações de crédito foram realizadas por mês?

SELECT
  DATE_TRUNC('month', application_date) AS mes,
  COUNT(*) AS qtd_solicitacoes
FROM credit_applications
GROUP BY mes
ORDER BY mes;



,mes,qtd_solicitacoes
0,2022-01-01,6
1,2022-02-01,37
2,2022-03-01,91
3,2022-04-01,119
4,2022-05-01,175
...,...,...
51,2026-04-01,298
52,2026-05-01,302
53,2026-06-01,311
54,2026-07-01,335


### A.2 — Qual foi o valor total solicitado e o valor total aprovado em cada mês?

In [5]:
run_query("a-2-valor-total-mes.sql")

-- Parte A, questão 2: Qual foi o valor total solicitado e o valor total aprovado em cada mês?

SELECT
  DATE_TRUNC('month', application_date) AS mes,
  SUM(requested_amount) AS valor_total_solicitado,
  SUM(approved_amount) AS valor_total_aprovado
FROM credit_applications
GROUP BY mes
ORDER BY mes;



,mes,valor_total_solicitado,valor_total_aprovado
0,2022-01-01,"46,278.06","16,923.02"
1,2022-02-01,"344,229.43","177,003.22"
2,2022-03-01,"853,988.83","412,513.91"
3,2022-04-01,"1,266,464.55","493,408.87"
4,2022-05-01,"1,698,467.99","667,228.84"
...,...,...,...
51,2026-04-01,"2,754,554.95","1,078,561.53"
52,2026-05-01,"2,924,685.15","1,301,047.63"
53,2026-06-01,"3,083,897.45","1,235,772.76"
54,2026-07-01,"3,129,766.83","1,222,494.78"


### A.3 — Qual foi a taxa de aprovação das propostas?

In [6]:
run_query("a-3-taxa-aprovacao.sql")

-- Parte A, questão 3: Qual foi a taxa de aprovação das propostas?

SELECT
  COUNT(*) FILTER (WHERE status = 'approved') AS qtd_aprovadas,
  COUNT(*) AS qtd_total,
  ROUND(COUNT(*) FILTER (WHERE status = 'approved') * 1.0 / COUNT(*), 4) AS taxa_aprovacao
FROM credit_applications;



,qtd_aprovadas,qtd_total,taxa_aprovacao
0,8874,16513,0.5374


### A.4 — Qual o ticket médio solicitado e o ticket médio aprovado?

In [7]:
run_query("a-4-ticket-medio.sql")

-- Parte A, questão 4: Qual o ticket médio solicitado e o ticket médio aprovado?

SELECT
  ROUND(AVG(requested_amount), 2) AS ticket_medio_solicitado,
  ROUND(AVG(approved_amount), 2) AS ticket_medio_aprovado
FROM credit_applications;



,ticket_medio_solicitado,ticket_medio_aprovado
0,"9,639.18","7,268.18"


## Parte B — Perfil e aprovação

### B.1 — Calcule a taxa de aprovação por faixa de renda.

In [8]:
run_query("b-1-aprovacao-faixa-renda.sql")

-- Parte B, questão 1: Calcule a taxa de aprovação por faixa de renda.

SELECT
  CASE
    WHEN cu.income IS NULL THEN 'Sem informação'
    WHEN cu.income < 2000 THEN 'Até R$2.000'
    WHEN cu.income < 4000 THEN 'R$2.000 a R$4.000'
    WHEN cu.income < 6000 THEN 'R$4.000 a R$6.000'
    WHEN cu.income < 10000 THEN 'R$6.000 a R$10.000'
    ELSE 'Acima de R$10.000'
  END AS faixa_renda,
  COUNT(*) FILTER (WHERE ca.status = 'approved') AS qtd_aprovadas,
  COUNT(*) AS qtd_total,
  ROUND(COUNT(*) FILTER (WHERE ca.status = 'approved') * 1.0 / COUNT(*), 4) AS taxa_aprovacao
FROM credit_applications ca
JOIN customers cu ON cu.customer_id = ca.customer_id
GROUP BY faixa_renda
ORDER BY MIN(COALESCE(cu.income, -1));



,faixa_renda,qtd_aprovadas,qtd_total,taxa_aprovacao
0,Sem informação,261,485,0.5381
1,Até R$2.000,1128,2075,0.5436
2,R$2.000 a R$4.000,4081,7615,0.5359
3,R$4.000 a R$6.000,2205,4129,0.5340
4,R$6.000 a R$10.000,1043,1925,0.5418
5,Acima de R$10.000,156,284,0.5493


### B.2 — Calcule a taxa de aprovação por faixa de risk_score.

In [9]:
run_query("b-2-aprovacao-faixa-risk-score.sql")

-- Parte B, questão 2: Calcule a taxa de aprovação por faixa de risk_score.

SELECT
  CASE
    WHEN risk_score IS NULL THEN 'Sem score'
    WHEN risk_score < 200 THEN '0-199'
    WHEN risk_score < 400 THEN '200-399'
    WHEN risk_score < 600 THEN '400-599'
    WHEN risk_score < 800 THEN '600-799'
    ELSE '800-1000'
  END AS faixa_risk_score,
  COUNT(*) FILTER (WHERE status = 'approved') AS qtd_aprovadas,
  COUNT(*) AS qtd_total,
  ROUND(COUNT(*) FILTER (WHERE status = 'approved') * 1.0 / COUNT(*), 4) AS taxa_aprovacao
FROM credit_applications
GROUP BY faixa_risk_score
ORDER BY MIN(COALESCE(risk_score, -1));



,faixa_risk_score,qtd_aprovadas,qtd_total,taxa_aprovacao
0,Sem score,256,490,0.5224
1,0-199,66,299,0.2207
2,200-399,986,2727,0.3616
3,400-599,3452,6916,0.4991
4,600-799,3199,4941,0.6474
5,800-1000,915,1140,0.8026


### B.3 — Qual percentual dos clientes aprovados efetivamente contratou um empréstimo?

In [10]:
run_query("b-3-pct-aprovados-contrataram.sql")

-- Parte B, questão 3: Qual percentual dos clientes aprovados efetivamente contratou um empréstimo?

WITH clientes_aprovados AS (
  SELECT DISTINCT customer_id
  FROM credit_applications
  WHERE status = 'approved'
)
SELECT
  COUNT(*) FILTER (WHERE EXISTS (SELECT 1 FROM loans l WHERE l.customer_id = ca.customer_id)) AS qtd_aprovados_que_contrataram,
  COUNT(*) AS qtd_clientes_aprovados,
  ROUND(
    COUNT(*) FILTER (WHERE EXISTS (SELECT 1 FROM loans l WHERE l.customer_id = ca.customer_id)) * 1.0 / COUNT(*),
    4
  ) AS pct_aprovados_que_contrataram
FROM clientes_aprovados ca;



,qtd_aprovados_que_contrataram,qtd_clientes_aprovados,pct_aprovados_que_contrataram
0,6077,6420,0.9466


### B.4 — Qual percentual do valor solicitado foi efetivamente aprovado?

In [11]:
run_query("b-4-pct-valor-aprovado.sql")

-- Parte B, questão 4: Qual percentual do valor solicitado foi efetivamente aprovado?

SELECT
  SUM(approved_amount) AS valor_total_aprovado,
  SUM(requested_amount) AS valor_total_solicitado,
  ROUND(SUM(approved_amount) / SUM(requested_amount), 4) AS pct_valor_aprovado
FROM credit_applications;



,valor_total_aprovado,valor_total_solicitado,pct_valor_aprovado
0,"64,497,867.11","159,171,703.68",0.4052


## Parte C — Inadimplência

### C.1 — Identifique quais contratos devem ser classificados como inadimplentes de acordo com a regra definida.

In [12]:
run_query("c-1-classificacao-inadimplencia.sql")

-- Parte C, questão 1: Identifique quais contratos devem ser classificados como inadimplentes de acordo com a regra definida.

SELECT
  loan_id,
  customer_id,
  contract_date,
  due_date,
  payment_date,
  loan_amount,
  amount_paid,
  CASE
    WHEN payment_date IS NOT NULL AND payment_date > due_date + INTERVAL 15 DAY THEN 'inadimplente'
    WHEN payment_date IS NOT NULL THEN 'adimplente'
    WHEN due_date + INTERVAL 15 DAY <= DATE '2026-08-23' THEN 'inadimplente'
    ELSE 'nao_maturado'
  END AS status_inadimplencia
FROM loans
ORDER BY contract_date;



,loan_id,customer_id,contract_date,due_date,payment_date,loan_amount,amount_paid,status_inadimplencia
0,2457,2961,2022-01-26,2022-02-25,2022-02-24,"5,642.77","6,115.36",adimplente
1,5256,6355,2022-01-27,2022-04-27,2022-04-26,"9,513.04","11,075.26",adimplente
2,1237,1489,2022-02-06,2022-05-07,2022-05-06,"1,568.85","1,721.02",adimplente
3,6086,7384,2022-02-10,2022-08-09,NaT,"3,110.92",NaN,inadimplente
4,7954,9697,2022-02-11,2022-08-10,2022-08-10,"11,751.09","11,963.91",adimplente
...,...,...,...,...,...,...,...,...
8213,2982,3573,2026-08-22,2026-09-21,NaT,"4,526.26",NaN,nao_maturado
8214,7933,9672,2026-08-23,2027-02-19,NaT,"1,765.55",NaN,nao_maturado
8215,6369,7724,2026-08-23,2027-08-18,NaT,"4,696.29",NaN,nao_maturado
8216,5882,7118,2026-08-23,2026-11-21,NaT,"5,194.23",NaN,nao_maturado


### C.2 — Calcule a taxa de inadimplência por mês de contratação.

In [13]:
run_query("c-2-inadimplencia-mes-contratacao.sql")

-- Parte C, questão 2: Calcule a taxa de inadimplência por mês de contratação.

WITH classificados AS (
  SELECT
    contract_date,
    CASE
      WHEN payment_date IS NOT NULL AND payment_date > due_date + INTERVAL 15 DAY THEN 'inadimplente'
      WHEN payment_date IS NOT NULL THEN 'adimplente'
      WHEN due_date + INTERVAL 15 DAY <= DATE '2026-08-23' THEN 'inadimplente'
      ELSE 'nao_maturado'
    END AS status_inadimplencia
  FROM loans
)
SELECT
  DATE_TRUNC('month', contract_date) AS mes_contratacao,
  COUNT(*) FILTER (WHERE status_inadimplencia = 'inadimplente') AS qtd_inadimplentes,
  COUNT(*) FILTER (WHERE status_inadimplencia IN ('adimplente', 'inadimplente')) AS qtd_classificados,
  ROUND(
    COUNT(*) FILTER (WHERE status_inadimplencia = 'inadimplente') * 1.0
      / COUNT(*) FILTER (WHERE status_inadimplencia IN ('adimplente', 'inadimplente')),
    4
  ) AS taxa_inadimplencia
FROM classificados
GROUP BY mes_contratacao
HAVING COUNT(*) FILTER (WHERE status_inadimplencia IN

,mes_contratacao,qtd_inadimplentes,qtd_classificados,taxa_inadimplencia
0,2022-03-01,11,53,0.2075
1,2022-04-01,15,54,0.2778
2,2022-05-01,8,81,0.0988
3,2022-06-01,23,105,0.2190
4,2022-07-01,20,122,0.1639
...,...,...,...,...
47,2026-02-01,20,117,0.1709
48,2026-03-01,22,115,0.1913
49,2026-04-01,14,96,0.1458
50,2026-05-01,3,61,0.0492


### C.3 — Calcule a taxa de inadimplência por faixa de risk_score.

In [14]:
run_query("c-3-inadimplencia-faixa-risk-score.sql")

-- Parte C, questão 3: Calcule a taxa de inadimplência por faixa de risk_score.

WITH classificados AS (
  SELECT
    loan_id,
    customer_id,
    contract_date,
    CASE
      WHEN payment_date IS NOT NULL AND payment_date > due_date + INTERVAL 15 DAY THEN 'inadimplente'
      WHEN payment_date IS NOT NULL THEN 'adimplente'
      WHEN due_date + INTERVAL 15 DAY <= DATE '2026-08-23' THEN 'inadimplente'
      ELSE 'nao_maturado'
    END AS status_inadimplencia
  FROM loans
),
com_score AS (
  SELECT
    c.*,
    (
      SELECT ca.risk_score
      FROM credit_applications ca
      WHERE ca.customer_id = c.customer_id
        AND ca.status = 'approved'
        AND ca.application_date <= c.contract_date
      ORDER BY ca.application_date DESC
      LIMIT 1
    ) AS risk_score_referencia
  FROM classificados c
  WHERE c.status_inadimplencia IN ('adimplente', 'inadimplente')
)
SELECT
  CASE
    WHEN risk_score_referencia IS NULL THEN 'Sem score'
    WHEN risk_score_referencia < 200 THEN '0-

,faixa_risk_score,qtd_inadimplentes,qtd_classificados,taxa_inadimplencia
0,Sem score,39,210,0.1857
1,0-199,15,56,0.2679
2,200-399,184,835,0.2204
3,400-599,492,2941,0.1673
4,600-799,315,2719,0.1159
5,800-1000,41,783,0.0524


### C.4 — Qual é o valor total da carteira inadimplente?

In [15]:
run_query("c-4-valor-carteira-inadimplente.sql")

-- Parte C, questão 4: Qual é o valor total da carteira inadimplente?

WITH classificados AS (
  SELECT
    loan_amount,
    CASE
      WHEN payment_date IS NOT NULL AND payment_date > due_date + INTERVAL 15 DAY THEN 'inadimplente'
      WHEN payment_date IS NOT NULL THEN 'adimplente'
      WHEN due_date + INTERVAL 15 DAY <= DATE '2026-08-23' THEN 'inadimplente'
      ELSE 'nao_maturado'
    END AS status_inadimplencia
  FROM loans
)
SELECT
  SUM(loan_amount) FILTER (WHERE status_inadimplencia = 'inadimplente') AS valor_carteira_inadimplente
FROM classificados;



,valor_carteira_inadimplente
0,"7,682,598.95"


### C.5 — Compare a inadimplência calculada por quantidade de contratos com a inadimplência calculada sobre o valor concedido.

In [16]:
run_query("c-5-inadimplencia-contrato-vs-valor.sql")

-- Parte C, questão 5: Compare a inadimplência calculada por quantidade de contratos com a
-- inadimplência calculada sobre o valor concedido.

WITH classificados AS (
  SELECT
    loan_amount,
    CASE
      WHEN payment_date IS NOT NULL AND payment_date > due_date + INTERVAL 15 DAY THEN 'inadimplente'
      WHEN payment_date IS NOT NULL THEN 'adimplente'
      WHEN due_date + INTERVAL 15 DAY <= DATE '2026-08-23' THEN 'inadimplente'
      ELSE 'nao_maturado'
    END AS status_inadimplencia
  FROM loans
)
SELECT
  ROUND(
    COUNT(*) FILTER (WHERE status_inadimplencia = 'inadimplente') * 1.0
      / COUNT(*) FILTER (WHERE status_inadimplencia IN ('adimplente', 'inadimplente')),
    4
  ) AS taxa_inadimplencia_por_contrato,
  ROUND(
    SUM(loan_amount) FILTER (WHERE status_inadimplencia = 'inadimplente')
      / SUM(loan_amount) FILTER (WHERE status_inadimplencia IN ('adimplente', 'inadimplente')),
    4
  ) AS taxa_inadimplencia_por_valor
FROM classificados;



,taxa_inadimplencia_por_contrato,taxa_inadimplencia_por_valor
0,0.1440,0.1439


## Parte D — SQL avançado

### D.1 — Para cada mês, calcule a taxa de aprovação e a variação em relação ao mês anterior.

In [17]:
run_query("d-1-variacao-taxa-aprovacao-mes.sql")

-- Parte D, questão 1: Para cada mês, calcule a taxa de aprovação e a variação em relação
-- ao mês anterior.

WITH por_mes AS (
  SELECT
    DATE_TRUNC('month', application_date) AS mes,
    COUNT(*) FILTER (WHERE status = 'approved') AS qtd_aprovadas,
    COUNT(*) AS qtd_total,
    ROUND(COUNT(*) FILTER (WHERE status = 'approved') * 1.0 / COUNT(*), 4) AS taxa_aprovacao
  FROM credit_applications
  GROUP BY mes
)
SELECT
  mes,
  taxa_aprovacao,
  ROUND(taxa_aprovacao - LAG(taxa_aprovacao) OVER (ORDER BY mes), 4) AS variacao_vs_mes_anterior,
  ROUND(LEAD(taxa_aprovacao) OVER (ORDER BY mes) - taxa_aprovacao, 4) AS variacao_vs_mes_seguinte,
  RANK() OVER (ORDER BY taxa_aprovacao DESC) AS rank_taxa_aprovacao
FROM por_mes
ORDER BY mes;



,mes,taxa_aprovacao,variacao_vs_mes_anterior,variacao_vs_mes_seguinte,rank_taxa_aprovacao
0,2022-01-01,0.5000,NaN,0.0946,49
1,2022-02-01,0.5946,0.0946,-0.0122,2
2,2022-03-01,0.5824,-0.0122,-0.0530,7
3,2022-04-01,0.5294,-0.0530,0.0135,34
4,2022-05-01,0.5429,0.0135,0.0088,28
...,...,...,...,...,...
51,2026-04-01,0.5000,-0.0732,0.0596,49
52,2026-05-01,0.5596,0.0596,-0.0580,17
53,2026-06-01,0.5016,-0.0580,0.0327,47
54,2026-07-01,0.5343,0.0327,-0.0219,30


### D.2 — Para cada cliente, identifique sua primeira solicitação de crédito.

In [18]:
run_query("d-2-primeira-solicitacao-cliente.sql")

-- Parte D, questão 2: Para cada cliente, identifique sua primeira solicitação de crédito.

SELECT
  customer_id,
  application_id AS primeira_solicitacao_id,
  application_date AS data_primeira_solicitacao,
  requested_amount,
  status
FROM credit_applications
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY application_date, application_id) = 1
ORDER BY customer_id;



,customer_id,primeira_solicitacao_id,data_primeira_solicitacao,requested_amount,status
0,1,1,2025-09-09,"8,691.33",rejected
1,2,3,2022-08-11,"1,025.93",rejected
2,3,4,2025-06-27,"8,745.70",approved
3,4,6,2022-12-30,"3,062.30",approved
4,5,7,2025-03-26,"3,199.04",approved
...,...,...,...,...,...
9445,9996,16505,2026-01-12,"9,033.03",rejected
9446,9997,16507,2025-05-15,"10,693.39",approved
9447,9998,16508,2024-01-28,"9,108.52",rejected
9448,9999,16509,2022-05-22,"7,106.12",rejected


### D.3 — Identifique clientes que tiveram uma solicitação rejeitada e posteriormente uma solicitação aprovada.

In [19]:
run_query("d-3-rejeitados-depois-aprovados.sql")

-- Parte D, questão 3: Identifique clientes que tiveram uma solicitação rejeitada e
-- posteriormente uma solicitação aprovada.

SELECT DISTINCT ca1.customer_id
FROM credit_applications ca1
WHERE ca1.status = 'rejected'
  AND EXISTS (
    SELECT 1
    FROM credit_applications ca2
    WHERE ca2.customer_id = ca1.customer_id
      AND ca2.status = 'approved'
      AND ca2.application_date > ca1.application_date
  )
ORDER BY ca1.customer_id;



,customer_id
0,6
1,12
2,15
3,17
4,43
...,...
1562,9932
1563,9944
1564,9966
1565,9980


### D.4 — Calcule o número de dias entre a primeira solicitação e a primeira aprovação de cada cliente.

In [20]:
run_query("d-4-dias-ate-primeira-aprovacao.sql")

-- Parte D, questão 4: Calcule o número de dias entre a primeira solicitação e a primeira
-- aprovação de cada cliente.

WITH por_cliente AS (
  SELECT
    customer_id,
    MIN(application_date) AS primeira_solicitacao,
    MIN(application_date) FILTER (WHERE status = 'approved') AS primeira_aprovacao
  FROM credit_applications
  GROUP BY customer_id
)
SELECT
  customer_id,
  primeira_solicitacao,
  primeira_aprovacao,
  DATE_DIFF('day', primeira_solicitacao, primeira_aprovacao) AS dias_ate_primeira_aprovacao
FROM por_cliente
WHERE primeira_aprovacao IS NOT NULL
ORDER BY customer_id;



,customer_id,primeira_solicitacao,primeira_aprovacao,dias_ate_primeira_aprovacao
0,3,2025-06-27,2025-06-27,0
1,4,2022-12-30,2022-12-30,0
2,5,2025-03-26,2025-03-26,0
3,6,2022-10-17,2022-12-05,49
4,8,2025-11-27,2025-11-27,0
...,...,...,...,...
6415,9993,2022-09-13,2022-09-13,0
6416,9994,2024-06-21,2024-06-21,0
6417,9995,2026-03-24,2026-03-24,0
6418,9997,2025-05-15,2025-05-15,0


### D.5 — Divida os clientes em cinco grupos de acordo com o risk_score e calcule a inadimplência de cada grupo.

Obs: o `score_max` de um grupo pode ser igual ao `score_min` do grupo seguinte. Isso acontece porque vários contratos têm o mesmo `risk_score`, e o `NTILE(5)` precisa dividir os contratos em grupos de tamanho igual: contratos com o mesmo score podem cair em grupos vizinhos (o desempate é feito pelo `loan_id`).

In [21]:
run_query("d-5-quintis-risk-score-inadimplencia.sql")

-- Parte D, questão 5: Divida os clientes em cinco grupos de acordo com o risk_score e
-- calcule a inadimplência de cada grupo.

WITH classificados AS (
  SELECT
    loan_id,
    customer_id,
    contract_date,
    CASE
      WHEN payment_date IS NOT NULL AND payment_date > due_date + INTERVAL 15 DAY THEN 'inadimplente'
      WHEN payment_date IS NOT NULL THEN 'adimplente'
      WHEN due_date + INTERVAL 15 DAY <= DATE '2026-08-23' THEN 'inadimplente'
      ELSE 'nao_maturado'
    END AS status_inadimplencia
  FROM loans
),
com_score AS (
  SELECT
    c.*,
    (
      SELECT ca.risk_score
      FROM credit_applications ca
      WHERE ca.customer_id = c.customer_id
        AND ca.status = 'approved'
        AND ca.application_date <= c.contract_date
      ORDER BY ca.application_date DESC
      LIMIT 1
    ) AS risk_score_referencia
  FROM classificados c
  WHERE c.status_inadimplencia IN ('adimplente', 'inadimplente')
),
com_quintil AS (
  SELECT
    *,
    NTILE(5) OVER (ORDER BY risk

,quintil_risk_score,score_min,score_max,qtd_inadimplentes,qtd_contratos,taxa_inadimplencia
0,1,64,451,302,1467,0.2059
1,2,451,548,249,1467,0.1697
2,3,548,632,218,1467,0.1486
3,4,632,732,177,1467,0.1207
4,5,732,1000,101,1466,0.0689
